# CRR Convergence Investigation

## 1. Objective

The objective of this investigation is to evaluate the convergence behaviour of the Cox–Ross–Rubinstein (CRR) binomial option pricing model by comparing its numerical prices against the analytical Black–Scholes solution.

The investigation examines:

1. Whether the CRR price converges to the Black–Scholes price as the number of time steps increases.
2. How the absolute pricing error changes as the tree becomes finer.
3. Whether odd and even numbers of time steps produce different convergence paths.
4. Whether convergence behaviour is affected by the option's moneyness.

Three European call options are considered:

- **In the money (ITM):** \(S_0 > K\)
- **At the money (ATM):** \(S_0 = K\)
- **Out of the money (OTM):** \(S_0 < K\)

All parameters other than the spot price are held constant so that differences in convergence can be attributed primarily to moneyness.

## 2. Methodology

The investigation was performed using the following procedure:

1. Define three European call options representing ITM, ATM and OTM cases.
2. Hold the strike price, maturity, volatility and risk-free rate constant across all three cases.
3. Calculate the analytical Black–Scholes price for each option.
4. Calculate the CRR price for each option using between 1 and 500 time steps.
5. Calculate the signed, absolute and relative pricing errors for each value of \(N\).
6. Plot:
   - CRR price against the Black–Scholes benchmark.
   - Absolute pricing error.
   - Absolute error on logarithmic axes.
   - Odd-step and even-step convergence separately.
7. Compare the convergence behaviour across the ITM, ATM and OTM cases.

The absolute pricing error is defined as

$$
E(N)
=
\left|
V_{\mathrm{CRR}}(N)-V_{\mathrm{BS}}
\right|,
$$

where $V_{\mathrm{CRR}}(N)$ is the CRR price calculated using $N$ time steps and $V_{\mathrm{BS}}$ is the corresponding Black–Scholes price.

The relative pricing error is

$$
E_{\mathrm{relative}}(N)
=
\frac{
\left|V_{\mathrm{CRR}}(N)-V_{\mathrm{BS}}\right|
}{
\left|V_{\mathrm{BS}}\right|
}.
$$

## 3. Test Parameters and Set Up

The following parameters were used:

| Parameter | ITM | ATM | OTM |
|---|---:|---:|---:|
| Option type | Call | Call | Call |
| Spot price, \(S_0\) | 120 | 100 | 80 |
| Strike price, \(K\) | 100 | 100 | 100 |
| Time to expiry, \(T\) | 1 year | 1 year | 1 year |
| Risk-free rate, \(r\) | 5% | 5% | 5% |
| Volatility, \(\sigma\) | 20% | 20% | 20% |
| CRR steps, \(N\) | 1–500 | 1–500 | 1–500 |

In [ ]:
import math

import matplotlib.pyplot as plt
import pandas as pd
from crr_convergence_utils import (
    investigate_crr_convergence,
    plot_scenario,plot_moneyness_comparison
)

from derivative_pricing.models.market import MarketData
from derivative_pricing.models.options import EuropeanOption, OptionType
from derivative_pricing.pricing.black_scholes_model import BlackScholesModel
from derivative_pricing.pricing.cox_ross_rubenstein_model import (
    CoxRossRubinsteinModel,
)

strike = 100.0
maturity = 1.0
risk_free_rate = 0.05
volatility = 0.20

step_counts = list(range(1, 501))

scenarios = {
    "ITM": {
        "spot": 120.0,
        "description": r"$S_0=120,\ K=100$",
    },
    "ATM": {
        "spot": 100.0,
        "description": r"$S_0=100,\ K=100$",
    },
    "OTM": {
        "spot": 80.0,
        "description": r"$S_0=80,\ K=100$",
    },
}

results: dict[str, pd.DataFrame] = {}

for scenario_name, scenario in scenarios.items():
    option = EuropeanOption(
        strike=strike,
        maturity=maturity,
        type=OptionType.CALL,
    )

    market = MarketData(
        spot=scenario["spot"],
        risk_free_rate=risk_free_rate,
        volatility=volatility,
    )

    results[scenario_name] = investigate_crr_convergence(
        option=option,
        market=market,
        step_counts=step_counts,
    )

## 4. Individual Convergence Results

The convergence plots for each moneyness case are shown below.

Each set of figures contains:

1. The CRR price relative to the Black–Scholes benchmark.
2. The absolute pricing error.
3. The absolute error on logarithmic axes.
4. The separate convergence paths for odd and even numbers of steps.

### 4.1 ITM Results
The in-the-money case has a spot price above the strike price:

$$
S_0=120>K=100.
$$

The plots are used to assess whether the CRR price approaches the Black–Scholes value, whether the pricing error decreases as \(N\) increases, and whether the odd and even subsequences approach the benchmark differently.

In [ ]:
plot_scenario(
    results["ITM"],
    scenario_name="In-the-money",
    description=scenarios["ITM"]["description"],
)

### 4.2 ATM Results
The at-the-money case has equal spot and strike prices:

$$
S_0=K=100.
$$

This case is particularly useful because the strike lies close to the centre of the terminal stock-price distribution, where the option payoff changes from zero to positive.

In [ ]:
plot_scenario(
    results["ATM"],
    scenario_name="At-the-money",
    description=scenarios["ATM"]["description"],
)

### 4.3 OTM results

The out-of-the-money case has a spot price below the strike price:

$$
S_0=80<K=100.
$$

The resulting option price is lower than in the ATM and ITM cases. Relative error should therefore also be considered because a small absolute error may represent a larger proportion of the option's value.

In [ ]:
plot_scenario(
    results["OTM"],
    scenario_name="Out-of-the-money",
    description=scenarios["OTM"]["description"],
)

In [ ]:
plot_moneyness_comparison(results)

In [ ]:
summary_steps = [10, 50, 100, 250, 500]

summary_rows = []

for scenario_name, df in results.items():
    selected = df[df["steps"].isin(summary_steps)]

    for row in selected.itertuples(index=False):
        summary_rows.append(
            {
                "scenario": scenario_name,
                "steps": row.steps,
                "black_scholes_price": row.black_scholes_price,
                "crr_price": row.crr_price,
                "absolute_error": row.absolute_error,
                "relative_error": row.relative_error,
            }
        )

summary_df = pd.DataFrame(summary_rows)

summary_df.style.format(
    {
        "black_scholes_price": "{:.8f}",
        "crr_price": "{:.8f}",
        "absolute_error": "{:.8f}",
        "relative_error": "{:.4%}",
    }
)

## Estimated Convergence Rate

If the pricing error follows

$$
E(N)\approx CN^{-\alpha},
$$

then

$$
\log E(N)
=
\log C-\alpha\log N.
$$

The convergence order, \(\alpha\), can therefore be estimated from the negative slope of a linear regression of \(\log E(N)\) against \(\log N\).

Since CRR prices commonly exhibit odd-even oscillation, the convergence order is estimated separately for odd and even values of \(N\).

In [ ]:
import numpy as np


def estimate_convergence_order(
    df: pd.DataFrame,
    minimum_steps: int = 50,
) -> float:
    filtered = df[
        (df["steps"] >= minimum_steps)
        & (df["absolute_error"] > 0)
    ]

    if len(filtered) < 2:
        return math.nan

    log_steps = np.log(filtered["steps"].to_numpy())
    log_errors = np.log(filtered["absolute_error"].to_numpy())

    slope, _ = np.polyfit(
        log_steps,
        log_errors,
        deg=1,
    )

    return float(-slope)


order_rows = []

for scenario_name, df in results.items():
    odd_df = df[df["steps"] % 2 == 1]
    even_df = df[df["steps"] % 2 == 0]

    order_rows.append(
        {
            "scenario": scenario_name,
            "odd_step_order": estimate_convergence_order(odd_df),
            "even_step_order": estimate_convergence_order(even_df),
        }
    )

order_df = pd.DataFrame(order_rows)

order_df.style.format(
    {
        "odd_step_order": "{:.4f}",
        "even_step_order": "{:.4f}",
    }
)

## Results and Discussion

For all three levels of moneyness, the CRR option price approached the corresponding Black–Scholes price as the number of time steps increased. This supports the expected limiting relationship

$$
\lim_{N\to\infty}V_{\mathrm{CRR}}(N)=V_{\mathrm{BS}}.
$$

The largest errors generally occurred when a relatively small number of time steps was used because a coarse binomial tree provides only a limited approximation of the continuous range of possible terminal stock prices. As the number of steps increased, the terminal stock-price distribution became more finely resolved and the CRR estimate moved closer to the Black–Scholes benchmark.

The convergence was not strictly monotonic. Odd and even values of \(N\) followed different convergence paths, producing an oscillating error pattern. In a CRR tree, the possible terminal stock prices are

$$
S_{N,j}=S_0u^jd^{N-j},
$$

where \(j\) is the number of upward movements. Changing \(N\) changes both the spacing and the location of these terminal nodes. As a result, the strike price may lie close to a terminal node for one value of \(N\), but between two terminal nodes for another.

This is important because the European call payoff

$$
\max(S_T-K,0)
$$

contains a kink at

$$
S_T=K.
$$

Below the strike, the payoff is zero, while above the strike it increases linearly. The payoff is therefore continuous but not differentiable at the strike. A discrete tree cannot represent this kink exactly unless the strike happens to align particularly well with the terminal nodes.

When the strike is represented more favourably by the terminal grid, the discrete expected payoff may be closer to the Black–Scholes value. When the strike falls less favourably between terminal nodes, the tree may either overestimate or underestimate the payoff. Moving from an odd number of steps to an even number changes this alignment, causing the pricing error to alternate rather than decrease smoothly.

The odd-even effect does not imply that the CRR model is unstable. The magnitude of the oscillation becomes smaller as \(N\) increases because the terminal nodes become more closely spaced. The tree therefore provides an increasingly accurate representation of the payoff near the strike, and both the odd-step and even-step subsequences converge toward the same Black–Scholes value.

The numerical results demonstrate this overall convergence, although the error does not fall at every individual increase in \(N\). For the ITM option, the absolute error fell from \(0.07940018\) at \(N=10\) to \(0.00245473\) at \(N=50\), but then increased to \(0.00864763\) at \(N=100\). It subsequently declined to \(0.00036531\) at \(N=250\) and \(0.00005259\) at \(N=500\). This temporary increase is consistent with the oscillatory convergence pattern rather than a failure of convergence.

For the ATM option, the convergence was more regular over the selected step counts. The absolute error decreased from \(0.19717453\) at \(N=10\) to \(0.03989203\) at \(N=50\), \(0.01997191\) at \(N=100\), \(0.00799486\) at \(N=250\), and \(0.00399844\) at \(N=500\). The corresponding relative error fell from \(1.8867\%\) to \(0.0383\%\).

The OTM option also displayed non-monotonic convergence. Its absolute error was \(0.01061019\) at \(N=10\), increased to \(0.02916260\) at \(N=50\), and then declined to \(0.00360865\), \(0.00219720\), and \(0.00077510\) at \(N=100\), \(250\), and \(500\), respectively. This again illustrates that a larger number of steps does not necessarily produce a smaller error for every individual value of \(N\), even though the general trend is toward the Black–Scholes solution.

At \(N=500\), the CRR prices and absolute errors were:

| Scenario | Black–Scholes price | CRR price | Absolute error | Relative error |
|---|---:|---:|---:|---:|
| ITM | 26.16904395 | 26.16909654 | 0.00005259 | 0.0002% |
| ATM | 10.45058357 | 10.44658514 | 0.00399844 | 0.0383% |
| OTM | 1.85941957 | 1.86019467 | 0.00077510 | 0.0417% |

The ITM option produced the smallest absolute and relative errors at \(N=500\). The ATM and OTM options had similar relative errors, although the ATM option had a larger absolute error. Since the OTM option had a much lower price, its relative error provides a more meaningful indication of accuracy than its absolute error alone.

The convergence order was estimated using

$$
E(N)\approx CN^{-\alpha},
$$

which implies

$$
\log E(N)=\log C-\alpha\log N.
$$

The estimated convergence orders were:

| Scenario | Odd-step order | Even-step order |
|---|---:|---:|
| ITM | 0.9515 | 1.0322 |
| ATM | 1.0006 | 0.9992 |
| OTM | 1.0741 | 1.0540 |

All estimated values were close to \(1\), indicating approximately first-order convergence:

$$
E(N)=O\left(\frac{1}{N}\right).
$$

This means that, on average, increasing the number of steps by a factor of two would be expected to reduce the error by approximately a factor of two, although the observed reduction for any particular pair of step counts may differ because of odd-even oscillation and terminal-node alignment.

The ATM option produced convergence orders almost exactly equal to \(1\) for both odd and even steps. The ITM option showed a slightly lower estimated order for odd steps and a slightly higher order for even steps, while both OTM estimates were slightly above \(1\). These small differences should not be interpreted as fundamentally different theoretical convergence rates. They are more likely caused by the finite range of \(N\) used for the regression and the oscillatory structure of the CRR pricing error.

Overall, the results show that the CRR model converges to the Black–Scholes solution for ITM, ATM and OTM options. The error decreases at an approximately first-order rate, but the convergence path is oscillatory rather than strictly monotonic because the location of the strike relative to the terminal stock-price grid changes as the number of steps changes.